# Capítulo 2. Preparación de datos reales

**Aprendizaje y Clasificación Automática con R**  
**Autor:** Jesús Gilberto Rodríguez Escobedo

Este cuaderno es **independiente y autónomo**: puede abrirse directamente sin ejecutar capítulos anteriores.

1. Ejecute primero la celda **Preparación automática y autónoma del capítulo**.
2. Después ejecute las celdas en orden.
3. Si Colab reinicia la sesión, vuelva a ejecutar desde la primera celda.

[Volver al índice de cuadernos Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/00-indice-colabs.ipynb)


In [ ]:
# Preparación automática y autónoma del capítulo
options(repos = c(CRAN = "https://cloud.r-project.org"))

paquetes_libro <- c(
  "ggplot2", "readr", "dplyr", "tidyr", "stringr", "data.table",
  "class", "rpart", "randomForest", "ranger", "e1071", "naivebayes",
  "neuralnet", "cluster", "caret", "factoextra", "scales", "plotly", "DT"
)
faltantes <- paquetes_libro[!vapply(paquetes_libro, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltantes)) install.packages(faltantes)

dir.create("datos/covid19/procesados", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/muestras", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/diccionarios", showWarnings = FALSE, recursive = TRUE)

archivos_colab <- c(
  "util_graficas.R" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/util_graficas.R",
  "datos/atus_ml_preparado.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/atus_ml_preparado.csv",
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  "datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz",
  "datos/covid19/diccionarios/diccionario_covid19_ml.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/diccionarios/diccionario_covid19_ml.csv"
)
for (destino in names(archivos_colab)) {
  if (!file.exists(destino)) download.file(archivos_colab[[destino]], destino, mode = "wb", quiet = TRUE)
}
stopifnot(all(file.exists(names(archivos_colab))))
source("util_graficas.R")
cat("Entorno autónomo listo. R:", R.version.string, "\n")


# Preparación de datos reales

La formulación matemática de **representación matemática de datos, probabilidad y estadística** se desarrolla con mayor profundidad
en los capítulos 1, 2 y 3 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de:

- localizar y descargar la base ATUS desde el portal oficial del INEGI;
- organizar los archivos dentro del proyecto Quarto;
- leer y revisar una base real en R;
- seleccionar y transformar variables;
- construir una variable respuesta binaria;
- guardar una base preparada para los capítulos de modelado;
- citar correctamente la fuente de los datos.

## Introducción

En la práctica profesional, los datos casi nunca llegan preparados para aplicar un algoritmo. Antes de modelar debemos conocer su procedencia, revisar su estructura, limpiar valores, transformar variables y documentar cada decisión.

En este libro utilizamos la **Estadística de Accidentes de Tránsito Terrestre en Zonas Urbanas y Suburbanas (ATUS)** del Instituto Nacional de Estadística y Geografía [@inegi_atus_2024]. Su objetivo es producir información anual sobre la siniestralidad del transporte terrestre en zonas de jurisdicción no federal, con desglose nacional, estatal y municipal.

Los datos originales son del INEGI. La limpieza, transformación, selección de variables, gráficas, modelos e interpretaciones de este libro son elaboración del autor. No constituyen resultados oficiales ni implican el aval del Instituto.

## Descargar la base ATUS desde el INEGI

La descarga debe realizarse desde el sitio oficial para conservar la procedencia y los metadatos del archivo.

1. Abra en su navegador el portal de **Accidentes de Tránsito Terrestre en Zonas Urbanas y Suburbanas (ATUS)**:

   <https://www.inegi.org.mx/programas/accidentes/>

2. Localice el apartado **Datos abiertos** o la opción de descarga correspondiente.
3. Seleccione el periodo anual que desea utilizar. Para reproducir los ejemplos de esta versión, elija **2024**.
4. Descargue el archivo en formato CSV. Es posible que el portal entregue un archivo comprimido en formato ZIP.
5. Descomprima el archivo descargado.
6. Identifique el CSV anual y cópielo en la carpeta `datos` del proyecto.
7. Para seguir exactamente los ejemplos, renómbrelo como:

```text
atus_2024.csv
```

La estructura mínima del proyecto debe quedar así:

```text
libro-machine-learning-r/
├── _quarto.yml
├── index.qmd
├── 02-preparacion-datos.qmd
├── datos/
│   └── atus_2024.csv
└── referencias.bib
```

Puede utilizar un año diferente. En ese caso, cambie el nombre del archivo o modifique la ruta utilizada en el código. Conviene anotar siempre el año de referencia porque los resultados pueden cambiar entre periodos.

## Cómo citar los datos

Los términos de libre uso del INEGI permiten utilizar, adaptar y publicar su información, pero requieren citar la fuente de origen [@inegi_terminos_libre_uso].

En el texto puede utilizarse una cita como esta:

> Los datos proceden de la Estadística de Accidentes de Tránsito Terrestre en Zonas Urbanas y Suburbanas del INEGI [@inegi_atus_2024].

Debajo de una tabla o gráfica sin transformaciones importantes:

> **Fuente:** INEGI, Estadística de Accidentes de Tránsito Terrestre en Zonas Urbanas y Suburbanas (ATUS), 2024.

Cuando exista procesamiento, agrupación, modelado o visualización propia:

> **Fuente:** Elaboración propia con datos del INEGI, Estadística de Accidentes de Tránsito Terrestre en Zonas Urbanas y Suburbanas (ATUS), 2024.

## Cargar paquetes


In [ ]:
library(readr)
library(dplyr)
library(ggplot2)
library(stringr)
source("util_graficas.R")


Se cargan paquetes para leer archivos CSV, transformar datos, trabajar con texto y crear gráficas. El archivo `util_graficas.R` contiene funciones visuales comunes para mantener un estilo uniforme en todo el libro.

## Localizar el archivo

En lugar de depender únicamente de un nombre fijo, el siguiente código busca archivos cuyo nombre comience con `atus` y termine en `.csv`.


In [ ]:
archivos_atus <- list.files(
  path = "datos",
  pattern = "^atus.*\\.csv$",
  full.names = TRUE,
  ignore.case = TRUE
)

# Preferir una base ATUS cruda cuando esté disponible.
# La base *_ml_preparado.csv es una salida de este mismo capítulo y
# no contiene necesariamente CLASACC.
archivos_crudos <- archivos_atus[
  !grepl("ml_preparado", basename(archivos_atus), ignore.case = TRUE)
]

archivos_atus


Si aparece al menos una ruta, R encontró un archivo ATUS. Si el resultado es `character(0)`, revise que el archivo esté descomprimido y guardado dentro de la carpeta `datos`.

## Leer el archivo CSV


In [ ]:
if (length(archivos_atus) == 0) {
  stop(
    paste(
      "No se encontró un archivo ATUS en la carpeta datos.",
      "Descárguelo desde el portal oficial del INEGI y descomprímalo."
    )
  )
}

ruta_atus <- if (length(archivos_crudos) > 0) {
  archivos_crudos[1]
} else {
  archivos_atus[1]
}

atus <- read_csv(
  file = ruta_atus,
  show_col_types = FALSE,
  locale = locale(encoding = "UTF-8")
)

message("Archivo leído: ", ruta_atus)


Se selecciona el primer archivo localizado y se lee con `read_csv()`. El argumento `show_col_types = FALSE` evita imprimir mensajes técnicos que distraen de los resultados principales.

## Primer vistazo a los datos


In [ ]:
resumen_dimensiones <- data.frame(
  filas = nrow(atus),
  columnas = ncol(atus)
)

resumen_dimensiones
head(atus)


Las filas representan registros de accidentes y las columnas describen sus características temporales, geográficas y operativas. Antes de modelar es indispensable confirmar que las variables esperadas estén disponibles.

## Normalizar los nombres de variables


In [ ]:
names(atus) <- toupper(names(atus))
names(atus)


Se convierten los nombres a mayúsculas para evitar errores por diferencias como `Mes`, `MES` o `mes`.

## Seleccionar variables de interés


In [ ]:
variables_interes <- c(
  "ANIO", "MES", "ID_ENTIDAD", "ID_MUNICIPIO",
  "ID_HORA", "ID_DIA", "DIASEMANA", "TIPACCID",
  "CAUSAACCI", "CLASACC"
)

variables_existentes <- intersect(
  variables_interes,
  names(atus)
)

variables_faltantes <- setdiff(
  variables_interes,
  names(atus)
)

atus_base <- atus |>
  select(all_of(variables_existentes))

variables_existentes
variables_faltantes


Los nombres y categorías pueden variar entre versiones. Si aparece una variable faltante, consulte los metadatos y el diccionario de datos descargados junto con la base antes de sustituirla.

## Crear la variable respuesta

El primer problema de clasificación distinguirá entre:

- **Con víctimas:** accidentes fatales o no fatales con personas lesionadas o fallecidas.
- **Solo daños:** accidentes con daños materiales sin víctimas registradas.


In [ ]:
if ("CLASACC" %in% names(atus_base)) {

  # Ruta A: base ATUS cruda del INEGI.
  atus_modelo <- atus_base |>
    mutate(
      CLASACC_TXT = str_to_lower(as.character(CLASACC)),
      accidente_con_victimas = case_when(
        str_detect(CLASACC_TXT, "sólo daños") ~ "Solo daños",
        str_detect(CLASACC_TXT, "solo daños") ~ "Solo daños",
        str_detect(CLASACC_TXT, "daños") ~ "Solo daños",
        str_detect(CLASACC_TXT, "no fatal") ~ "Con víctimas",
        str_detect(CLASACC_TXT, "fatal") ~ "Con víctimas",
        TRUE ~ NA_character_
      )
    ) |>
    filter(!is.na(accidente_con_victimas))

} else if ("ACCIDENTE_CON_VICTIMAS" %in% names(atus)) {

  # Ruta B: base ya preparada incluida con el libro.
  # Después de normalizar nombres, accidente_con_victimas aparece en mayúsculas.
  atus_modelo <- atus |>
    transmute(
      MES,
      ID_HORA,
      DIASEMANA,
      TIPACCID,
      CAUSAACCI,
      accidente_con_victimas = as.character(ACCIDENTE_CON_VICTIMAS)
    ) |>
    mutate(
      accidente_con_victimas = case_when(
        str_to_lower(accidente_con_victimas) %in% c(
          "con víctimas", "con victimas"
        ) ~ "Con víctimas",
        str_to_lower(accidente_con_victimas) %in% c(
          "solo daños", "sólo daños"
        ) ~ "Solo daños",
        TRUE ~ NA_character_
      )
    ) |>
    filter(!is.na(accidente_con_victimas))

  message(
    "Se utilizó datos/atus_ml_preparado.csv como respaldo para el render. ",
    "Para reproducir desde cero la preparación, use la base ATUS cruda del INEGI."
  )

} else {
  stop(
    paste(
      "No se encontró CLASACC ni ACCIDENTE_CON_VICTIMAS.",
      "Revise el diccionario y la versión de la base ATUS."
    )
  )
}

table(atus_modelo$accidente_con_victimas)


La variable `CLASACC` se transforma en una respuesta binaria. `case_when()` asigna una nueva categoría según el texto encontrado y los registros que no pueden clasificarse se excluyen temporalmente.

## Preparar las variables finales


In [ ]:
atus_ml <- atus_modelo |>
  mutate(
    MES = as.factor(MES),
    ID_HORA = as.character(ID_HORA),
    DIASEMANA = as.factor(DIASEMANA),
    TIPACCID = as.factor(TIPACCID),
    CAUSAACCI = as.factor(CAUSAACCI),
    accidente_con_victimas = factor(
      accidente_con_victimas,
      levels = c("Con víctimas", "Solo daños")
    )
  ) |>
  select(
    accidente_con_victimas,
    MES,
    ID_HORA,
    DIASEMANA,
    TIPACCID,
    CAUSAACCI
  ) |>
  na.omit()

data.frame(
  filas = nrow(atus_ml),
  columnas = ncol(atus_ml)
)


La base final queda lista para el análisis exploratorio y el modelado. Cada fila representa un accidente y cada columna una variable predictora o la respuesta que se desea clasificar.

## Guardar la base preparada


In [ ]:
if (!dir.exists("datos")) {
  dir.create("datos")
}

write_csv(
  atus_ml,
  "datos/atus_ml_preparado.csv"
)


Guardar la base preparada evita repetir toda la limpieza en los capítulos siguientes y ayuda a mantener reproducible el flujo de trabajo.

## Lista de comprobación

Antes de continuar, verifique que:

- el archivo proviene del portal oficial del INEGI;
- el año de los datos está documentado;
- el CSV se encuentra dentro de `datos`;
- las variables usadas existen en la versión descargada;
- la transformación de `CLASACC` corresponde con su diccionario;
- las tablas y gráficas incluyen la fuente;
- el archivo `atus_ml_preparado.csv` se creó correctamente.

## Resumen del capítulo

En este capítulo descargamos y documentamos una base real del INEGI, verificamos su ubicación, seleccionamos variables, construimos una respuesta binaria y guardamos una base preparada para aprendizaje automático.

## Materiales complementarios del capítulo
Estos recursos permiten repasar la preparación de datos reales mediante una presentación, una infografía y un video explicativo.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual del proceso de descarga, revisión, limpieza y preparación de datos. | [Ver en YouTube](https://www.youtube.com/watch?v=bmv-tDPZGjQ) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-02/capitulo-02-preparacion-datos-reales.pdf) | [Descargar PDF](recursos/capitulo-02/capitulo-02-preparacion-datos-reales.pdf){download="capitulo-02-preparacion-datos-reales.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-02/capitulo-02-preparacion-datos-reales.pptx) | [Descargar PPTX](recursos/capitulo-02/capitulo-02-preparacion-datos-reales.pptx){download="capitulo-02-preparacion-datos-reales.pptx"} |
| Infografía | Resumen visual de las etapas de preparación de datos. | [Ver infografía](recursos/capitulo-02/capitulo-02-preparacion-datos-reales-infografia.png) | [Descargar PNG](recursos/capitulo-02/capitulo-02-preparacion-datos-reales-infografia.png){download="capitulo-02-preparacion-datos-reales-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/02-preparacion-datos.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 2](recursos/capitulo-02/capitulo-02-preparacion-datos-reales-infografia.png)](recursos/capitulo-02/capitulo-02-preparacion-datos-reales-infografia.png)

**Video del capítulo:** <https://www.youtube.com/watch?v=bmv-tDPZGjQ>

La presentación PDF, el archivo editable y la infografía pueden descargarse desde la versión web del libro.

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor. El texto del libro y sus archivos fuente constituyen la referencia principal.

Este video forma parte de la lista oficial del curso **Aprendizaje y Clasificación Automática con R**.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)

## Laboratorio interactivo: preparar datos

Explora valores faltantes, imputación y estandarización.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

Permite introducir valores faltantes, imputarlos y estandarizar las variables.

## Caso aplicado B: preparación de datos COVID-19

En esta segunda ruta aplicada utilizamos una muestra nacional de **50 000
registros confirmados de COVID-19 en México durante 2022**. El año fue
seleccionado después de comparar los cierres históricos que conservan un
esquema homologable.

La base preparada se encuentra en:

```text
datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz
```

### Lectura de la base


In [ ]:
library(readr)
library(dplyr)

covid <- read_csv(
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  show_col_types = FALSE
)

dim(covid)
names(covid)


### Variables binarias y códigos especiales

En la base preparada, las variables clínicas de tipo sí/no fueron
recodificadas así:

| Valor | Interpretación |
|---:|---|
| 0 | No |
| 1 | Sí |
| `NA` | No especificado, se ignora o no aplica |

La variable `TIPO_PACIENTE` quedó codificada como:

| Valor | Interpretación |
|---:|---|
| 0 | Ambulatorio |
| 1 | Hospitalizado |

La variable objetivo `MURIO` se derivó de `FECHA_DEF`:

| Valor | Interpretación |
|---:|---|
| 0 | Sin defunción registrada |
| 1 | Defunción registrada |

### Revisión de valores faltantes


In [ ]:
faltantes_covid <- covid |>
  summarise(
    across(
      everything(),
      ~ sum(is.na(.x))
    )
  ) |>
  tidyr::pivot_longer(
    cols = everything(),
    names_to = "variable",
    values_to = "faltantes"
  ) |>
  arrange(desc(faltantes))

head(faltantes_covid, 10)


### Número de comorbilidades

La variable `NUM_COMORBILIDADES` resume la presencia de:

- diabetes;
- EPOC;
- asma;
- inmunosupresión;
- hipertensión;
- enfermedad cardiovascular;
- obesidad;
- enfermedad renal crónica;
- tabaquismo.


In [ ]:
covid |>
  count(NUM_COMORBILIDADES) |>
  arrange(NUM_COMORBILIDADES)


Estos datos se emplean con fines educativos. Los registros administrativos
pueden contener sesgos, valores desconocidos y diferencias de cobertura. Los
modelos construidos con esta base no sustituyen una valoración médica.
